# PSID Classification Sweep

Runs `--phases classification` for all 8 PSID configs (z-as-behavior and z-as-neural × 4 sessions).  
CPU-only notebook — 4 parallel workers, no GPU needed.

**Dataset required:** `giedriusmirklys/psid-data`

In [ ]:
# ── 1. Install packages ───────────────────────────────────────────────────────
import subprocess, sys


def pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", *args], capture_output=True, text=True
    )
    if result.returncode != 0:
        print("INSTALL FAILED:", " ".join(args))
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
    else:
        print("OK:", " ".join(args))


pip("--no-deps", "--ignore-requires-python", "PSID==1.2.6")
pip("polars>=1.0.0")

In [ ]:
# ── 2. Paths ──────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

INPUT = Path("/kaggle/input/psid-data")
WORK = Path("/kaggle/working")

sys.path.insert(0, str(INPUT / "src"))
os.chdir(str(WORK))

import PSID

print("PSID: imported OK")

In [ ]:
# ── 3. Symlink result data into working dir ───────────────────────────────────
for src_variant in sorted(INPUT.glob("results/psid/psid_z-as-*_dbs_*")):
    dst_variant = WORK / "results" / "psid" / src_variant.name
    dst_variant.mkdir(parents=True, exist_ok=True)
    for item in src_variant.iterdir():
        link = dst_variant / item.name
        if not link.exists():
            os.symlink(item, link)

dst_setups = WORK / "training" / "setups"
dst_setups.mkdir(parents=True, exist_ok=True)
for yaml_file in (INPUT / "training/setups").glob("psid_PDI*.yaml"):
    link = dst_setups / yaml_file.name
    if not link.exists():
        os.symlink(yaml_file, link)

(WORK / "logs" / "psid").mkdir(parents=True, exist_ok=True)

variant_count = len(list((WORK / "results" / "psid").iterdir()))
print(f"Symlinked {variant_count} variant dirs")

In [ ]:
# ── 4. Run classification — 2 workers via fork (CPU-only, no GPU) ─────────────
import multiprocessing
import concurrent.futures
import traceback

SESSIONS = ["PDI1_S2", "PDI1_S4", "PDI4_S2", "PDI4_S3"]
MODES = ["z-as-behavior", "z-as-neural"]
YAML_DIR = WORK / "training" / "setups"

# fork: child inherits already-imported deps, avoids re-importing heavy packages
# safe here because there is no GPU/TF state in parent


def _run_one(args):
    session, mode, yaml_dir, work_dir = args
    import traceback
    from pathlib import Path
    from utils.config import get_config
    from utils.logger import setup_logging
    from training.pipelines._base import FrameworkPipeline

    yaml_path = Path(yaml_dir) / f"psid_{session}_{mode}.yaml"
    if not yaml_path.exists():
        return (session, mode, "SKIP")
    try:
        config = get_config(str(yaml_path))
        log = setup_logging(
            f"psid_{session}_{mode}",
            Path(work_dir) / "logs" / "psid" / f"{session}_{mode}.log",
        )
        FrameworkPipeline(config, log, phases=("classification",)).run()
        return (session, mode, "OK")
    except Exception as exc:
        traceback.print_exc()
        return (session, mode, str(exc))


args_list = [(s, m, str(YAML_DIR), str(WORK)) for m in MODES for s in SESSIONS]
ctx = multiprocessing.get_context("fork")
errors = []

with concurrent.futures.ProcessPoolExecutor(max_workers=2, mp_context=ctx) as ex:
    for session, mode, result in ex.map(_run_one, args_list):
        print(f"{result}: {session}/{mode}")
        if result not in ("OK", "SKIP"):
            errors.append((session, mode, result))

print(f"\nFinished. {len(errors)} errors.")
for s, m, e in errors:
    print(f"  FAILED {s}/{m}: {e}")

In [ ]:
# ── 5. List output sweep parquets ─────────────────────────────────────────────
total_rows = 0
import polars as pl

for cls_dir in sorted(
    (WORK / "results" / "psid").glob("psid_z-as-*_dbs_both/classification")
):
    parquets = sorted(cls_dir.glob("sweep_*.parquet"))
    if not parquets:
        print(f"  MISSING: {cls_dir.parent.name}")
        continue
    final = [
        p for p in parquets if not any(t in p.name for t in ["predictions", "forecast"])
    ]
    p = final[-1] if final else parquets[-1]
    df = pl.read_parquet(p)
    total_rows += len(df)
    size_kb = p.stat().st_size // 1024
    print(f"  {cls_dir.parent.name}: {len(df)} rows, {size_kb} KB -> {p.name}")

print(f"\nTotal rows: {total_rows}")